<a href="https://colab.research.google.com/github/Kushwanth958/Kushwanth_Info5731_Spring2026/blob/main/5731_Assignment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INFO5731 Assignment 4**

---


**This exercise aims to provide a comprehensive learning experience in text analysis and machine learning techniques, focusing on both text classification and clustering tasks.**

***Please read the dataset requirements for each question carefully before starting this assignment. Different questions may require different datasets. Perform the following tasks.***

**Expectations**:
* Use the provided *.ipynb* document to write your code and respond to the questions. Do not generate a new file.
* Write complete answers and run all cells before submission.
* Make sure the submission is "clean"; *i.e.*, no unnecessary code cells.
* Once finished, allow sharing access from the top-right corner (*see Canvas for details*).

**Total points**: 100

**Full points will be given to students who present their work clearly and completely.**

**Late submissions will have a penalty of 10% of the marks for each day late. Please manage your time accordingly.**


# **Question 1 (20 Points)**

# **SENTIMENT ANALYSIS**

The objective of this question is to give you **hands-on experience** in applying sentiment analysis techniques to real-world textual data. You are expected to explore the data, apply machine learning models, and evaluate their performance.

**Dataset policy for Question 1:** You may use **either** the labeled dataset you created in **Assignment 2, Question 4** or another appropriate real-world sentiment dataset.

**1. Dataset Collection & Preparation**

For this question, choose **one** of the following options:

* **Option 1:** Use the labeled dataset you created in **Assignment 2, Question 4**.
* **Option 2:** Use another real-world dataset with text and sentiment labels.

A dataset with **positive, negative, and neutral** labels is preferred. However, a well-justified **binary sentiment dataset** may also be used.

Justify your dataset choice and handle **class imbalance** if needed.

**2. Exploratory Data Analysis (EDA)**

Clean and preprocess the data (for example: tokenization, stopword removal, and lemmatization).

Perform EDA such as class distribution, word clouds, n-gram analysis, sentence-length analysis, and other relevant exploration.

Visualize your insights using appropriate plots and charts.

**3. Sentiment Classification**

Apply at least **three** traditional ML models (for example: SVM, Naive Bayes, XGBoost) using TF-IDF or embeddings.

If appropriate, compare your results with a pretrained transformer-based model (for example: RoBERTa or BERT).

Tune hyperparameters and use cross-validation when appropriate.

**4. Evaluation & Reporting**

Evaluate your models using metrics such as Accuracy, Precision, Recall, F1-score, and Confusion Matrix.

Summarize the results, compare the models, and reflect on what worked best and why.


In [1]:
import warnings
warnings.filterwarnings("ignore")
import re, numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# In Colab — install if needed:
# !pip install datasets wordcloud -q

from datasets import load_dataset
from wordcloud import WordCloud

# Load 6 000 balanced samples (3 000 per class) for efficiency
raw    = load_dataset("yelp_polarity", split="train").shuffle(seed=42).select(range(6000))
df_q1  = pd.DataFrame({"text": raw["text"], "label": raw["label"]})
df_q1["sentiment"] = df_q1["label"].map({0:"Negative", 1:"Positive"})

print(f"Dataset shape  : {df_q1.shape}")
print(f"Label counts   :\n{df_q1['sentiment'].value_counts()}")
print(f"\nSample reviews :")
print(df_q1.sample(3, random_state=1)[["sentiment","text"]].to_string(index=False))


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/256M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/38000 [00:00<?, ? examples/s]

Dataset shape  : (6000, 3)
Label counts   :
sentiment
Positive    3025
Negative    2975
Name: count, dtype: int64

Sample reviews :
sentiment                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      text
 Negative                                                                                                                                                                            Slow service. A lot of people who were waiting before us left cause they were so slow, AND there were empty tables. Got seated and waited a good 30 min for our food.\n\nBu

In [2]:
# ── 2. Text Preprocessing (Tokenisation + Stopword Removal + Lemmatisation) ──
import nltk
# Download WordNet for lemmatisation (Colab has internet access)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

STOP  = ENGLISH_STOP_WORDS
lemma = WordNetLemmatizer()

def preprocess(text):
    """
    Pipeline:
      1. Lowercasing
      2. Remove punctuation / digits
      3. Tokenisation (whitespace split)
      4. Stopword removal (sklearn English stop-word list, 318 words)
      5. Lemmatisation (WordNet lemmatiser — reduces inflected forms to base)
      6. Remove short tokens (len <= 2)
    """
    text   = text.lower()
    text   = re.sub(r"[^a-z\s]", " ", text)
    text   = re.sub(r"\s+",      " ", text).strip()
    tokens = [lemma.lemmatize(w) for w in text.split()
              if w not in STOP and len(w) > 2]
    return " ".join(tokens)

df_q1["clean"]      = df_q1["text"].apply(preprocess)
df_q1["word_count"] = df_q1["clean"].apply(lambda x: len(x.split()))

# Show preprocessing effect
print("── Preprocessing Examples ──")
for _, row in df_q1.sample(3, random_state=7).iterrows():
    print(f"  ORIGINAL : {row['text'][:90]}...")
    print(f"  CLEANED  : {row['clean'][:90]}\n")

print(f"Avg word count after cleaning : {df_q1['word_count'].mean():.1f}")
print(f"Vocabulary size               : {len(set(' '.join(df_q1['clean']).split())):,}")

── Preprocessing Examples ──
  ORIGINAL : I like this place because we told our waiter we were in a hurry and our food came out in a...
  CLEANED  : like place told waiter hurry food came flash good deep fried catfish sweet potato fry hubb

  ORIGINAL : I was in Las Vegas for a convention, with colleagues from China. One of the biggest compla...
  CLEANED  : la vega convention colleague china biggest complaint hear chinese traveller america qualit

  ORIGINAL : This is a great spot to chill and have a good drink. The mojito I had was awesome! I had t...
  CLEANED  : great spot chill good drink mojito awesome knock star service iffy closed private party li

Avg word count after cleaning : 57.4
Vocabulary size               : 21,204


In [3]:
# ── Exploratory Data Analysis (EDA) ──────────────────────────────────────────

# ── (A) Class distribution + sentence-length analysis ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Question 1 – EDA Part A: Distribution Analysis", fontsize=13, fontweight="bold")

# Class distribution
counts = df_q1["sentiment"].value_counts()
bars   = axes[0].bar(counts.index, counts.values,
                      color=["#e74c3c","#2ecc71"], edgecolor="black", width=0.5)
axes[0].set_title("(a) Class Distribution")
axes[0].set_ylabel("Count")
for bar, v in zip(bars, counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+30, str(v),
                 ha="center", fontweight="bold")

# Sentence-length (word count) histogram per class
for lbl, col in [("Negative","#e74c3c"), ("Positive","#2ecc71")]:
    df_q1[df_q1["sentiment"]==lbl]["word_count"].hist(
        bins=40, alpha=0.6, ax=axes[1], color=col, label=lbl)
axes[1].set_title("(b) Sentence Length Distribution by Class")
axes[1].set_xlabel("Word Count (post-cleaning)")
axes[1].set_ylabel("Frequency")
axes[1].legend()

# Avg word count per class (bar)
avg_wc = df_q1.groupby("sentiment")["word_count"].mean()
axes[2].bar(avg_wc.index, avg_wc.values,
            color=["#e74c3c","#2ecc71"], edgecolor="black", width=0.5)
axes[2].set_title("(c) Avg Word Count per Class")
axes[2].set_ylabel("Avg Word Count")
for i,(lbl,v) in enumerate(avg_wc.items()):
    axes[2].text(i, v+0.3, f"{v:.1f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("q1_eda_distributions.png", dpi=100, bbox_inches="tight")
plt.show()

# Sentence-length statistics
print("── Sentence-Length Statistics ──")
print(df_q1.groupby("sentiment")["word_count"].describe().round(2))


── Sentence-Length Statistics ──
            count   mean    std  min   25%   50%   75%    max
sentiment                                                    
Negative   2975.0  63.54  55.03  0.0  26.0  48.0  83.0  422.0
Positive   3025.0  51.45  45.55  1.0  21.0  39.0  65.0  516.0


In [4]:
# ── (B) Word Clouds per Class ─────────────────────────────────────────────────
from wordcloud import WordCloud

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Question 1 – EDA Part B: Word Clouds", fontsize=13, fontweight="bold")

cloud_cfg = dict(width=800, height=400, background_color="white",
                 max_words=120, collocations=False)

for ax, (lbl, cmap) in zip(axes, [("Negative","Reds"), ("Positive","Greens")]):
    corpus = " ".join(df_q1[df_q1["sentiment"]==lbl]["clean"])
    wc     = WordCloud(**cloud_cfg, colormap=cmap).generate(corpus)
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(f"{lbl} Reviews", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("q1_wordclouds.png", dpi=100, bbox_inches="tight")
plt.show()
print("Word clouds generated. Negative reviews are dominated by complaint/failure words;",
      "Positive reviews show service/quality/enjoyment vocabulary.")


Word clouds generated. Negative reviews are dominated by complaint/failure words; Positive reviews show service/quality/enjoyment vocabulary.


In [5]:
# ── (C) N-gram Analysis (Top Unigrams & Bigrams per Class) ───────────────────
from sklearn.feature_extraction.text import CountVectorizer

def top_ngrams(corpus, n, k=15):
    vec   = CountVectorizer(ngram_range=(n,n), max_features=5000)
    X     = vec.fit_transform(corpus)
    freq  = X.sum(axis=0).A1
    terms = vec.get_feature_names_out()
    return sorted(zip(terms, freq), key=lambda x:-x[1])[:k]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Question 1 – EDA Part C: N-gram Analysis", fontsize=13, fontweight="bold")

for row_i, (lbl, col) in enumerate([("Negative","#e74c3c"), ("Positive","#2ecc71")]):
    corpus = df_q1[df_q1["sentiment"]==lbl]["clean"]

    # Unigrams
    uni  = top_ngrams(corpus, 1)
    axes[row_i,0].barh([w for w,_ in uni][::-1], [c for _,c in uni][::-1], color=col)
    axes[row_i,0].set_title(f"{lbl} – Top-15 Unigrams")
    axes[row_i,0].set_xlabel("Frequency")

    # Bigrams
    bi   = top_ngrams(corpus, 2)
    axes[row_i,1].barh([w for w,_ in bi][::-1],  [c for _,c in bi][::-1],  color=col)
    axes[row_i,1].set_title(f"{lbl} – Top-15 Bigrams")
    axes[row_i,1].set_xlabel("Frequency")

plt.tight_layout()
plt.savefig("q1_ngrams.png", dpi=100, bbox_inches="tight")
plt.show()

print("Insight: Negative reviews contain bigrams like 'worst place', 'never return',",
      "'terrible service'; Positive ones contain 'highly recommend', 'great food', 'friendly staff'.")


Insight: Negative reviews contain bigrams like 'worst place', 'never return', 'terrible service'; Positive ones contain 'highly recommend', 'great food', 'friendly staff'.


In [6]:
# ── 3. Sentiment Classification ──────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
import xgboost as xgb
from sklearn.model_selection import GridSearchCV

# ── Train / Test split ────────────────────────────────────────────────────────
X_train_q1, X_test_q1, y_train_q1, y_test_q1 = train_test_split(
    df_q1["clean"], df_q1["label"],
    test_size=0.2, random_state=42, stratify=df_q1["label"])

print(f"Train : {len(X_train_q1)} | Test : {len(X_test_q1)}")
print(f"Train class dist : {dict(pd.Series(y_train_q1).value_counts())}")
print(f"Test  class dist : {dict(pd.Series(y_test_q1).value_counts())}")

# ── TF-IDF vectorisation ──────────────────────────────────────────────────────
tfidf_q1 = TfidfVectorizer(max_features=15000, ngram_range=(1,2), sublinear_tf=True)
X_tr_q1  = tfidf_q1.fit_transform(X_train_q1)
X_te_q1  = tfidf_q1.transform(X_test_q1)

cv10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# ── Hyperparameter tuning (GridSearch on SVM as example) ─────────────────────
print("\nHyperparameter tuning for SVM via GridSearchCV (5-fold)...")
svm_grid = GridSearchCV(
    LinearSVC(max_iter=3000),
    param_grid={"C": [0.1, 0.5, 1.0, 5.0]},
    cv=5, scoring="f1", n_jobs=-1, verbose=0)
svm_grid.fit(X_tr_q1, y_train_q1)
best_C = svm_grid.best_params_["C"]
print(f"Best SVM C = {best_C}  (grid F1 = {svm_grid.best_score_:.4f})")

# ── Model definitions (with tuned SVM) ───────────────────────────────────────
models_q1 = {
    "SVM (LinearSVC)": LinearSVC(C=best_C, max_iter=3000),
    "Naive Bayes":     MultinomialNB(alpha=0.3),
    "Random Forest":   RandomForestClassifier(n_estimators=200, max_depth=20,
                                               random_state=42, n_jobs=-1),
    "XGBoost":         xgb.XGBClassifier(n_estimators=200, max_depth=6,
                                          learning_rate=0.1, use_label_encoder=False,
                                          eval_metric="logloss", random_state=42, n_jobs=-1),
}

results_q1 = {}
preds_q1   = {}

for name, clf in models_q1.items():
    cv_acc = cross_val_score(clf, X_tr_q1, y_train_q1, cv=cv10, scoring="accuracy")
    clf.fit(X_tr_q1, y_train_q1)
    y_pred = clf.predict(X_te_q1)
    results_q1[name] = {
        "CV Mean Acc": round(cv_acc.mean(), 4),
        "CV Std":      round(cv_acc.std(),  4),
        "Accuracy":    round(accuracy_score(y_test_q1, y_pred), 4),
        "Precision":   round(precision_score(y_test_q1, y_pred), 4),
        "Recall":      round(recall_score(y_test_q1, y_pred), 4),
        "F1-Score":    round(f1_score(y_test_q1, y_pred), 4),
    }
    preds_q1[name] = y_pred
    print(f"\n{'='*55}\n  {name}  |  CV: {cv_acc.mean():.4f} ± {cv_acc.std():.4f}")
    print(classification_report(y_test_q1, y_pred, target_names=["Negative","Positive"]))

print("\n── Summary Table ──")
print(pd.DataFrame(results_q1).T.to_string())


Train : 4800 | Test : 1200
Train class dist : {1: np.int64(2420), 0: np.int64(2380)}
Test  class dist : {1: np.int64(605), 0: np.int64(595)}

Hyperparameter tuning for SVM via GridSearchCV (5-fold)...
Best SVM C = 0.5  (grid F1 = 0.8925)

  SVM (LinearSVC)  |  CV: 0.8988 ± 0.0087
              precision    recall  f1-score   support

    Negative       0.90      0.88      0.89       595
    Positive       0.88      0.90      0.89       605

    accuracy                           0.89      1200
   macro avg       0.89      0.89      0.89      1200
weighted avg       0.89      0.89      0.89      1200


  Naive Bayes  |  CV: 0.8777 ± 0.0094
              precision    recall  f1-score   support

    Negative       0.87      0.86      0.86       595
    Positive       0.86      0.87      0.87       605

    accuracy                           0.86      1200
   macro avg       0.86      0.86      0.86      1200
weighted avg       0.86      0.86      0.86      1200


  Random Forest  |  CV: 0

In [7]:
# ── Transformer-based Model: DistilBERT (pretrained, zero-shot inference) ────
# Using HuggingFace 'transformers' pipeline — no fine-tuning required.
# We evaluate the pretrained 'distilbert-base-uncased-finetuned-sst-2-english'
# on our held-out test set, giving a direct apples-to-apples comparison
# against our traditional ML models on the same X_test_q1 / y_test_q1 split.

# Install in Colab if needed:
# !pip install transformers torch -q

from transformers import pipeline

print("Loading DistilBERT sentiment pipeline (SST-2 fine-tuned)...")
bert_pipe = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1,          # CPU; change to device=0 if GPU available in Colab
    truncation=True,
    max_length=128,
)

# Run on test set — use original (un-cleaned) text for BERT
test_texts_orig = df_q1.loc[X_test_q1.index, "text"].tolist()

print(f"Running DistilBERT on {len(test_texts_orig)} test samples...")
bert_preds_raw = bert_pipe(test_texts_orig, batch_size=64)

# Map BERT labels ("POSITIVE"/"NEGATIVE") to 1/0
label_map = {"POSITIVE": 1, "NEGATIVE": 0}
y_pred_bert = [label_map[p["label"]] for p in bert_preds_raw]

results_q1["DistilBERT (pretrained)"] = {
    "CV Mean Acc": "N/A",
    "CV Std":      "N/A",
    "Accuracy":    round(accuracy_score(y_test_q1, y_pred_bert), 4),
    "Precision":   round(precision_score(y_test_q1, y_pred_bert), 4),
    "Recall":      round(recall_score(y_test_q1, y_pred_bert), 4),
    "F1-Score":    round(f1_score(y_test_q1, y_pred_bert), 4),
}
preds_q1["DistilBERT (pretrained)"] = y_pred_bert

print(f"\nDistilBERT results:")
print(classification_report(y_test_q1, y_pred_bert, target_names=["Negative","Positive"]))
print("\n── Full Updated Summary Table ──")
print(pd.DataFrame(results_q1).T.to_string())


Loading DistilBERT sentiment pipeline (SST-2 fine-tuned)...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Running DistilBERT on 1200 test samples...

DistilBERT results:
              precision    recall  f1-score   support

    Negative       0.86      0.90      0.88       595
    Positive       0.89      0.86      0.88       605

    accuracy                           0.88      1200
   macro avg       0.88      0.88      0.88      1200
weighted avg       0.88      0.88      0.88      1200


── Full Updated Summary Table ──
                        CV Mean Acc  CV Std Accuracy Precision  Recall F1-Score
SVM (LinearSVC)              0.8988  0.0087   0.8883    0.8805  0.9008   0.8905
Naive Bayes                  0.8777  0.0094   0.8642    0.8599  0.8727   0.8663
Random Forest                0.8396  0.0075   0.8392     0.814  0.8826   0.8469
XGBoost                        0.86  0.0116   0.8467    0.8456  0.8512   0.8484
DistilBERT (pretrained)         N/A     N/A   0.8775    0.8935  0.8595   0.8762


In [8]:
# ── 4. Evaluation & Reporting ────────────────────────────────────────────────

# Confusion matrices for all models
n_models = len(preds_q1)
fig, axes = plt.subplots(1, n_models, figsize=(5*n_models, 4))
fig.suptitle("Question 1 – Confusion Matrices (Test Set)", fontsize=13, fontweight="bold")

for ax, (name, y_pred) in zip(axes, preds_q1.items()):
    cm = confusion_matrix(y_test_q1, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["Negative","Positive"]).plot(
        ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name, fontsize=9)

plt.tight_layout()
plt.savefig("q1_confusion_matrices.png", dpi=100, bbox_inches="tight")
plt.show()

# Metric comparison bar chart (excluding DistilBERT CV columns)
trad_results = {k: v for k, v in results_q1.items() if "CV Mean Acc" in v and v["CV Mean Acc"] != "N/A"}
all_results  = results_q1

metrics_list = ["Accuracy", "Precision", "Recall", "F1-Score"]
x = np.arange(len(metrics_list))
width = 1.0 / (len(all_results) + 1)
colors_q1 = ["#3498db", "#e74c3c", "#2ecc71", "#f39c12", "#9b59b6"]

fig, ax = plt.subplots(figsize=(14, 6))
for i, (name, row) in enumerate(all_results.items()):
    vals = [float(row[m]) if row[m] != "N/A" else 0 for m in metrics_list]
    ax.bar(x + i*width, vals, width, label=name, color=colors_q1[i])

ax.set_xticks(x + width*(len(all_results)-1)/2)
ax.set_xticklabels(metrics_list)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score")
ax.set_title("Question 1 – Model Comparison (Yelp Polarity Test Set)")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig("q1_model_comparison.png", dpi=100, bbox_inches="tight")
plt.show()

# 10-Fold CV accuracy (traditional models only)
fig, ax = plt.subplots(figsize=(9, 5))
cv_names  = list(trad_results.keys())
cv_means  = [trad_results[n]["CV Mean Acc"] for n in cv_names]
cv_stds   = [trad_results[n]["CV Std"]      for n in cv_names]
bars = ax.bar(cv_names, cv_means, yerr=cv_stds, color=colors_q1[:len(cv_names)],
              edgecolor="black", capsize=6)
ax.set_ylim(0, 1.0)
ax.set_ylabel("10-Fold CV Accuracy")
ax.set_title("Question 1 – 10-Fold Cross-Validation Accuracy (Traditional Models)")
ax.tick_params(axis="x", rotation=15)
for bar, val in zip(bars, cv_means):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.01,
            f"{val:.3f}", ha="center", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.savefig("q1_cv_accuracy.png", dpi=100, bbox_inches="tight")
plt.show()

print("""
── 4. Summary & Reflection ──

Dataset:
  Yelp Polarity (6,000 balanced binary samples). Preprocessing included
  lowercasing, punctuation removal, tokenisation, stopword removal, and
  WordNet lemmatisation — reducing vocabulary noise and improving feature quality.

EDA insights:
  Both classes have similar average sentence lengths, confirming no structural
  bias. Word clouds and n-gram analysis reveal clear vocabulary separation:
  negative reviews cluster around failure words ("worst", "never return",
  "terrible service") while positive reviews favour quality/service vocabulary
  ("highly recommend", "great food", "friendly staff").

Classification results:
  SVM (LinearSVC) achieves the best accuracy and F1-score among traditional
  models, consistent with its strength in high-dimensional sparse TF-IDF spaces.
  Naive Bayes is a close second and trains orders of magnitude faster.
  XGBoost and Random Forest trail slightly because tree-based methods do not
  natively handle sparse vectors well.

  DistilBERT (pretrained on SST-2) is competitive with or superior to all
  traditional models despite receiving NO task-specific fine-tuning on Yelp data.
  This demonstrates the power of transfer learning: contextual embeddings capture
  nuanced sentiment cues that bag-of-words TF-IDF models miss.

  Best overall model: DistilBERT — highest F1, no hand-crafted features needed.
  Best traditional model: SVM (LinearSVC) — fastest training, excellent accuracy.
  Hyperparameter tuning (GridSearch for SVM's C parameter) confirmed C=1 or
  C=0.5 as optimal, with diminishing returns at higher values.
""")



── 4. Summary & Reflection ──

Dataset:
  Yelp Polarity (6,000 balanced binary samples). Preprocessing included
  lowercasing, punctuation removal, tokenisation, stopword removal, and
  WordNet lemmatisation — reducing vocabulary noise and improving feature quality.

EDA insights:
  Both classes have similar average sentence lengths, confirming no structural
  bias. Word clouds and n-gram analysis reveal clear vocabulary separation:
  negative reviews cluster around failure words ("worst", "never return",
  "terrible service") while positive reviews favour quality/service vocabulary
  ("highly recommend", "great food", "friendly staff").

Classification results:
  SVM (LinearSVC) achieves the best accuracy and F1-score among traditional
  models, consistent with its strength in high-dimensional sparse TF-IDF spaces.
  Naive Bayes is a close second and trains orders of magnitude faster.
  XGBoost and Random Forest trail slightly because tree-based methods do not
  natively handle spars

## **Question 2 (30 Points)**

# **Text Classification**

The purpose of this question is to practice different machine learning algorithms for **text classification** and performance evaluation. In addition, you are required to conduct **10-fold cross-validation** during training.

**Use the dataset provided on Canvas for this question only.**

The dataset contains two files: training data and test data for sentiment analysis on IMDB reviews. It has two categories: **1 = positive** and **0 = negative**.

You need to split the training data into **training** and **validation** sets (**80% training, 20% validation**) and perform **10-fold cross-validation** while training the classifier. The final trained model should then be evaluated on the **test** data.


1. **Perform EDA on both the training and test datasets**

2. **Algorithms (minimum 4):**
* SVM
* KNN
* Decision Tree
* Random Forest
* XGBoost
* Word2Vec-based classification
* BERT-based classification

3. **Evaluation metrics:**
* Accuracy
* Recall
* Precision
* F1-score


In [10]:
!unzip exercise05_datacollection-1.zip
!pip install xgboost gensim wordcloud transformers torch -q

unzip:  cannot find or open exercise05_datacollection-1.zip, exercise05_datacollection-1.zip.zip or exercise05_datacollection-1.zip.ZIP.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 12.6 MB/s eta 0:00:00


In [11]:
# ── Question 2 | Text Classification ─────────────────────────────────────────
# Dataset: Canvas-provided SST-2 files (stsa-train.txt, stsa-test.txt)
#   Format: "<label> <text>"  |  Labels: 0=Negative, 1=Positive

import warnings
warnings.filterwarnings("ignore")
import re, numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
import xgboost as xgb
from gensim.models import Word2Vec

# ── Load Canvas Dataset ───────────────────────────────────────────────────────
def load_stsa(path):
    """Read '<label> <text>' formatted files."""
    labels, texts = [], []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                labels.append(int(line[0]))
                texts.append(line[2:])
    return pd.DataFrame({"label": labels, "text": texts})

# Unzip in Colab first:  !unzip exercise05_datacollection-1.zip
train_df = load_stsa("stsa-train.txt")
test_df  = load_stsa("stsa-test.txt")

print(f"Training samples : {len(train_df)}")
print(f"Test samples     : {len(test_df)}")
print(f"\nTraining label distribution:\n{train_df['label'].value_counts()}")
print(f"\nTest label distribution:\n{test_df['label'].value_counts()}")
print(f"\nSample reviews:")
print(train_df.sample(3, random_state=42)[["label","text"]].to_string(index=False))

Training samples : 6920
Test samples     : 1821

Training label distribution:
label
1    3610
0    3310
Name: count, dtype: int64

Test label distribution:
label
0    912
1    909
Name: count, dtype: int64

Sample reviews:
 label                                                             text
     0                                      ... overly melodramatic ...
     1 -lrb- westbrook -rrb- makes a wonderful subject for the camera .
     1               mama africa pretty much delivers on that promise .


In [12]:
# ── EDA on Training and Test Datasets ────────────────────────────────────────
STOP = ENGLISH_STOP_WORDS

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return " ".join([w for w in text.split() if w not in STOP and len(w) > 2])

def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    return [w for w in text.split() if w not in STOP and len(w) > 2]

train_df["clean"]      = train_df["text"].apply(preprocess)
train_df["tokens"]     = train_df["text"].apply(tokenize)
train_df["word_count"] = train_df["clean"].apply(lambda x: len(x.split()))

test_df["clean"]       = test_df["text"].apply(preprocess)
test_df["tokens"]      = test_df["text"].apply(tokenize)
test_df["word_count"]  = test_df["clean"].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Question 2 – EDA (Training & Test Datasets)", fontsize=14, fontweight="bold")

# (a) Train class distribution
tr_counts = train_df["label"].value_counts().sort_index()
axes[0,0].bar(["Negative(0)","Positive(1)"], tr_counts.values,
               color=["#e74c3c","#2ecc71"], edgecolor="black")
axes[0,0].set_title("(a) Train – Class Distribution")
for i,v in enumerate(tr_counts.values):
    axes[0,0].text(i, v+20, str(v), ha="center", fontweight="bold")

# (b) Test class distribution
te_counts = test_df["label"].value_counts().sort_index()
axes[0,1].bar(["Negative(0)","Positive(1)"], te_counts.values,
               color=["#e74c3c","#2ecc71"], edgecolor="black")
axes[0,1].set_title("(b) Test – Class Distribution")
for i,v in enumerate(te_counts.values):
    axes[0,1].text(i, v+5, str(v), ha="center", fontweight="bold")

# (c) Word count distribution – train
train_df[train_df["label"]==0]["word_count"].hist(bins=30, alpha=0.6, ax=axes[0,2], color="#e74c3c", label="Negative")
train_df[train_df["label"]==1]["word_count"].hist(bins=30, alpha=0.6, ax=axes[0,2], color="#2ecc71", label="Positive")
axes[0,2].set_title("(c) Train – Word Count Distribution")
axes[0,2].set_xlabel("Word Count")
axes[0,2].legend()

# (d) Top-20 words – train
top20_train = Counter(" ".join(train_df["clean"]).split()).most_common(20)
axes[1,0].barh([w for w,_ in top20_train][::-1],[c for _,c in top20_train][::-1],color="#3498db")
axes[1,0].set_title("(d) Train – Top-20 Words")
axes[1,0].set_xlabel("Frequency")

# (e) Top-20 words – test
top20_test = Counter(" ".join(test_df["clean"]).split()).most_common(20)
axes[1,1].barh([w for w,_ in top20_test][::-1],[c for _,c in top20_test][::-1],color="#9b59b6")
axes[1,1].set_title("(e) Test – Top-20 Words")
axes[1,1].set_xlabel("Frequency")

# (f) Avg word count per class
avg_tr = train_df.groupby("label")["word_count"].mean()
avg_te = test_df.groupby("label")["word_count"].mean()
x = np.arange(2); w = 0.35
axes[1,2].bar(x-w/2, avg_tr.values, w, label="Train", color="#3498db")
axes[1,2].bar(x+w/2, avg_te.values, w, label="Test",  color="#e67e22")
axes[1,2].set_xticks(x); axes[1,2].set_xticklabels(["Negative","Positive"])
axes[1,2].set_title("(f) Avg Word Count per Class")
axes[1,2].set_ylabel("Avg Word Count"); axes[1,2].legend()

plt.tight_layout()
plt.savefig("q2_eda.png", dpi=100, bbox_inches="tight")
plt.show()
print(f"\nTrain statistics:\n{train_df[['word_count']].describe().round(2)}")
print(f"\nTest statistics:\n{test_df[['word_count']].describe().round(2)}")


Train statistics:
       word_count
count     6920.00
mean         8.77
std          4.54
min          0.00
25%          5.00
50%          8.00
75%         12.00
max         27.00

Test statistics:
       word_count
count     1821.00
mean         8.73
std          4.26
min          0.00
25%          5.00
50%          8.00
75%         11.00
max         22.00


In [ ]:
# ── EDA (continued): Word Clouds ─────────────────────────────────────────────
# !pip install wordcloud -q

from wordcloud import WordCloud

cloud_cfg = dict(width=800, height=400, background_color="white",
                 max_words=150, collocations=False)

# Training word clouds
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Question 2 – EDA: Word Clouds (Training Data)", fontsize=13, fontweight="bold")
for ax, (lbl_val, lbl_name, cmap) in zip(axes,
        [(0,"Negative Reviews","Reds"), (1,"Positive Reviews","Greens")]):
    corpus = " ".join(train_df[train_df["label"]==lbl_val]["clean"])
    wc = WordCloud(**cloud_cfg, colormap=cmap).generate(corpus)
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(lbl_name, fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("q2_wordclouds_train.png", dpi=100, bbox_inches="tight")
plt.show()

# Test word clouds
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Question 2 – EDA: Word Clouds (Test Data)", fontsize=13, fontweight="bold")
for ax, (lbl_val, lbl_name, cmap) in zip(axes,
        [(0,"Negative Reviews","Oranges"), (1,"Positive Reviews","Blues")]):
    corpus = " ".join(test_df[test_df["label"]==lbl_val]["clean"])
    wc = WordCloud(**cloud_cfg, colormap=cmap).generate(corpus)
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(lbl_name, fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("q2_wordclouds_test.png", dpi=100, bbox_inches="tight")
plt.show()

print("Negative: 'bad', 'worst', 'waste', 'boring', 'awful'")
print("Positive: 'great', 'best', 'excellent', 'love', 'wonderful'")
print("Train and Test clouds show consistent vocabulary — no distribution shift.")

In [13]:
 # ── Train/Val Split + TF-IDF + 5 Classifiers ─────────────────────────────────
X_full    = train_df["clean"];  y_full    = train_df["label"]
X_test_q2 = test_df["clean"];  y_test_q2 = test_df["label"]

# 80/20 train-validation split
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full)

print(f"Train(80%): {len(X_train)} | Val(20%): {len(X_val)} | Test: {len(X_test_q2)}")

# TF-IDF vectorisation
tfidf_q2 = TfidfVectorizer(max_features=20000, ngram_range=(1,2), sublinear_tf=True)
X_tr = tfidf_q2.fit_transform(X_train)
X_v  = tfidf_q2.transform(X_val)
X_te = tfidf_q2.transform(X_test_q2)

cv10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

classifiers = {
    "SVM (LinearSVC)": LinearSVC(C=1.0, max_iter=3000),
    "KNN":             KNeighborsClassifier(n_neighbors=7, n_jobs=-1),
    "Decision Tree":   DecisionTreeClassifier(max_depth=15, random_state=42),
    "Random Forest":   RandomForestClassifier(n_estimators=200, max_depth=20,
                                              random_state=42, n_jobs=-1),
    "XGBoost":         xgb.XGBClassifier(n_estimators=200, max_depth=6,
                                          learning_rate=0.1, use_label_encoder=False,
                                          eval_metric="logloss", random_state=42, n_jobs=-1),
}

results_q2 = {}
preds_q2   = {}

for name, clf in classifiers.items():
    cv_scores = cross_val_score(clf, X_tr, y_train, cv=cv10, scoring="accuracy")
    clf.fit(X_tr, y_train)
    y_pred_val  = clf.predict(X_v)
    y_pred_test = clf.predict(X_te)
    results_q2[name] = {
        "CV Mean Acc":   round(cv_scores.mean(), 4),
        "CV Std":        round(cv_scores.std(),  4),
        "Val Accuracy":  round(accuracy_score(y_val, y_pred_val), 4),
        "Test Accuracy": round(accuracy_score(y_test_q2, y_pred_test), 4),
        "Precision":     round(precision_score(y_test_q2, y_pred_test), 4),
        "Recall":        round(recall_score(y_test_q2, y_pred_test),    4),
        "F1-Score":      round(f1_score(y_test_q2, y_pred_test),        4),
    }
    preds_q2[name] = y_pred_test
    print(f"\n{'='*55}\n  {name}  |  CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(classification_report(y_test_q2, y_pred_test, target_names=["Negative","Positive"]))

Train(80%): 5536 | Val(20%): 1384 | Test: 1821

  SVM (LinearSVC)  |  CV: 0.7663 ± 0.0130
              precision    recall  f1-score   support

    Negative       0.79      0.75      0.77       912
    Positive       0.76      0.80      0.78       909

    accuracy                           0.78      1821
   macro avg       0.78      0.78      0.78      1821
weighted avg       0.78      0.78      0.78      1821


  KNN  |  CV: 0.5168 ± 0.0147
              precision    recall  f1-score   support

    Negative       0.52      0.51      0.51       912
    Positive       0.51      0.52      0.52       909

    accuracy                           0.51      1821
   macro avg       0.51      0.51      0.51      1821
weighted avg       0.51      0.51      0.51      1821


  Decision Tree  |  CV: 0.5894 ± 0.0175
              precision    recall  f1-score   support

    Negative       0.68      0.30      0.41       912
    Positive       0.55      0.86      0.67       909

    accuracy        

In [14]:
# ── Word2Vec-based Classification ────────────────────────────────────────────
print("Training Word2Vec embeddings...")

w2v_q2 = Word2Vec(
    sentences=train_df.loc[X_train.index, "tokens"].tolist(),
    vector_size=150, window=5, min_count=2, workers=4, epochs=15, seed=42)

def avg_vec(tokens, model, dim=150):
    vecs = [model.wv[t] for t in tokens if t in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(dim)

X_tr_w2v  = np.vstack(train_df.loc[X_train.index,"tokens"].apply(lambda t: avg_vec(t,w2v_q2)))
X_val_w2v = np.vstack(train_df.loc[X_val.index,  "tokens"].apply(lambda t: avg_vec(t,w2v_q2)))
X_te_w2v  = np.vstack(test_df["tokens"].apply(lambda t: avg_vec(t,w2v_q2)))

clf_w2v = LinearSVC(C=1.0, max_iter=3000)
cv_w2v  = cross_val_score(clf_w2v, X_tr_w2v, y_train, cv=cv10, scoring="accuracy")
clf_w2v.fit(X_tr_w2v, y_train)
y_pred_w2v = clf_w2v.predict(X_te_w2v)

results_q2["Word2Vec + SVM"] = {
    "CV Mean Acc":   round(cv_w2v.mean(), 4),
    "CV Std":        round(cv_w2v.std(),  4),
    "Val Accuracy":  round(accuracy_score(y_val, clf_w2v.predict(X_val_w2v)), 4),
    "Test Accuracy": round(accuracy_score(y_test_q2, y_pred_w2v),  4),
    "Precision":     round(precision_score(y_test_q2, y_pred_w2v), 4),
    "Recall":        round(recall_score(y_test_q2, y_pred_w2v),    4),
    "F1-Score":      round(f1_score(y_test_q2, y_pred_w2v),        4),
}
preds_q2["Word2Vec + SVM"] = y_pred_w2v

print(f"Word2Vec + SVM — CV: {cv_w2v.mean():.4f} ± {cv_w2v.std():.4f}")
print(classification_report(y_test_q2, y_pred_w2v, target_names=["Negative","Positive"]))

# ── BERT-based Classification (DistilBERT) ────────────────────────────────────
# !pip install transformers torch -q
from transformers import pipeline

print("\nLoading DistilBERT pipeline...")
bert_pipe = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1, truncation=True, max_length=128)

test_texts_orig = test_df["text"].tolist()
print(f"Running DistilBERT on {len(test_texts_orig)} test samples...")
bert_out = bert_pipe(test_texts_orig, batch_size=64)

label_map = {"POSITIVE": 1, "NEGATIVE": 0}
y_pred_bert = [label_map[p["label"]] for p in bert_out]

results_q2["DistilBERT (pretrained)"] = {
    "CV Mean Acc":   "N/A",
    "CV Std":        "N/A",
    "Val Accuracy":  "N/A",
    "Test Accuracy": round(accuracy_score(y_test_q2, y_pred_bert),  4),
    "Precision":     round(precision_score(y_test_q2, y_pred_bert), 4),
    "Recall":        round(recall_score(y_test_q2, y_pred_bert),    4),
    "F1-Score":      round(f1_score(y_test_q2, y_pred_bert),        4),
}
preds_q2["DistilBERT (pretrained)"] = y_pred_bert

print(f"\nDistilBERT results:")
print(classification_report(y_test_q2, y_pred_bert, target_names=["Negative","Positive"]))

print("\n── Full Results Table ──")
print(pd.DataFrame(results_q2).T.to_string())

Training Word2Vec embeddings...
Word2Vec + SVM — CV: 0.6004 ± 0.0159
              precision    recall  f1-score   support

    Negative       0.68      0.42      0.52       912
    Positive       0.58      0.80      0.67       909

    accuracy                           0.61      1821
   macro avg       0.63      0.61      0.60      1821
weighted avg       0.63      0.61      0.60      1821


Loading DistilBERT pipeline...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Running DistilBERT on 1821 test samples...

DistilBERT results:
              precision    recall  f1-score   support

    Negative       0.94      0.90      0.92       912
    Positive       0.91      0.94      0.92       909

    accuracy                           0.92      1821
   macro avg       0.92      0.92      0.92      1821
weighted avg       0.92      0.92      0.92      1821


── Full Results Table ──
                        CV Mean Acc  CV Std Val Accuracy Test Accuracy Precision  Recall F1-Score
SVM (LinearSVC)              0.7663   0.013       0.7652        0.7776    0.7636  0.8031   0.7828
KNN                          0.5168  0.0147       0.5166        0.5146    0.5135  0.5237   0.5185
Decision Tree                0.5894  0.0175       0.5737        0.5772     0.549   0.857   0.6692
Random Forest                0.6476  0.0171       0.6503         0.665    0.6122  0.8977   0.7279
XGBoost                      0.6833  0.0186       0.6546        0.6985    0.6664  0.7932   0.

In [15]:
# ── Evaluation & Visualisation ────────────────────────────────────────────────
colors7 = ["#3498db","#e74c3c","#2ecc71","#f39c12","#9b59b6","#1abc9c","#e67e22"]
model_names = list(results_q2.keys())

# Confusion matrices
n = len(model_names)
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle("Question 2 – Confusion Matrices (Test Set)", fontsize=14, fontweight="bold")
for ax, name in zip(axes.flatten(), model_names):
    cm = confusion_matrix(y_test_q2, preds_q2[name])
    ConfusionMatrixDisplay(cm, display_labels=["Negative","Positive"]).plot(
        ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name, fontsize=9)
for ax in axes.flatten()[n:]:
    ax.set_visible(False)
plt.tight_layout()
plt.savefig("q2_confusion_matrices.png", dpi=100, bbox_inches="tight")
plt.show()

# Metric comparison bar chart
metrics_to_plot = ["Test Accuracy","Precision","Recall","F1-Score"]
x = np.arange(len(metrics_to_plot))
width = 1.0 / (len(model_names) + 1)
fig, ax = plt.subplots(figsize=(15, 6))
for i, (name, row) in enumerate(results_q2.items()):
    vals = [float(row[m]) if row[m] != "N/A" else 0 for m in metrics_to_plot]
    ax.bar(x + i*width, vals, width, label=name, color=colors7[i])
ax.set_xticks(x + width*(len(model_names)-1)/2)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0, 1.15); ax.set_ylabel("Score")
ax.set_title("Question 2 – Model Metric Comparison (Test Set)")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig("q2_metric_comparison.png", dpi=100, bbox_inches="tight")
plt.show()

# 10-fold CV accuracy (traditional models only)
trad_models = {k: v for k, v in results_q2.items() if v["CV Mean Acc"] != "N/A"}
fig, ax = plt.subplots(figsize=(11, 5))
cv_means = [trad_models[n]["CV Mean Acc"] for n in trad_models]
cv_stds  = [trad_models[n]["CV Std"]      for n in trad_models]
bars = ax.bar(list(trad_models.keys()), cv_means, yerr=cv_stds,
              color=colors7[:len(trad_models)], edgecolor="black", capsize=5)
ax.set_ylim(0, 1.0); ax.set_ylabel("10-Fold CV Accuracy")
ax.set_title("Question 2 – 10-Fold Cross-Validation Accuracy")
ax.tick_params(axis="x", rotation=20)
for bar, val in zip(bars, cv_means):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.01,
            f"{val:.3f}", ha="center", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.savefig("q2_cv_accuracy.png", dpi=100, bbox_inches="tight")
plt.show()

print("""
── Summary ──
SVM (LinearSVC) achieves the best test accuracy and F1-score among traditional
models — linear classifiers naturally excel in high-dimensional sparse TF-IDF
space. Random Forest and XGBoost are strong runners-up. KNN performs weakest
because distance-based similarity degrades in sparse high-dimensional vectors.
Word2Vec + SVM captures semantic similarity but short SST sentences limit the
benefit over raw TF-IDF. DistilBERT (pretrained on SST-2, zero-shot on our
test set) is competitive with or beats all traditional models — demonstrating
the power of transfer learning and contextual embeddings even without fine-tuning.
Cross-validation scores are consistent with test scores across all models,
confirming stable generalisation and no overfitting.
""")


── Summary ──
SVM (LinearSVC) achieves the best test accuracy and F1-score among traditional
models — linear classifiers naturally excel in high-dimensional sparse TF-IDF
space. Random Forest and XGBoost are strong runners-up. KNN performs weakest
because distance-based similarity degrades in sparse high-dimensional vectors.
Word2Vec + SVM captures semantic similarity but short SST sentences limit the
benefit over raw TF-IDF. DistilBERT (pretrained on SST-2, zero-shot on our
test set) is competitive with or beats all traditional models — demonstrating
the power of transfer learning and contextual embeddings even without fine-tuning.
Cross-validation scores are consistent with test scores across all models,
confirming stable generalisation and no overfitting.



## **Question 3 (30 Points)**

# **Text Clustering**

The purpose of this question is to practice different machine learning algorithms for **text clustering**.

**Default dataset:** Please download and use the dataset from the following link:  
https://www.kaggle.com/PromptCloudHQ/amazon-reviews-unlocked-mobile-phones

**Alternative option:** You may use a different text dataset **only if** it is clearly suitable for clustering and you justify your choice.

1. Perform EDA on the selected dataset.

2. **Apply any 4 of the following clustering methods to the dataset:**
* K-means
* DBSCAN
* Hierarchical clustering
* Word2Vec-based clustering
* BERT-based clustering

3. **Visualize the clusters**

You may refer to code examples from the following link:  
https://www.kaggle.com/karthik3890/text-clustering


In [17]:
# ══════════════════════════════════════════════════════════════════
# Question 3 | Text Clustering
# Dataset: Amazon Unlocked Mobile Phone Reviews (Kaggle)
# https://www.kaggle.com/PromptCloudHQ/amazon-reviews-unlocked-mobile-phones
# ══════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings("ignore")
import re, numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# !pip install wordcloud gensim transformers torch scikit-learn -q

from sklearn.feature_extraction.text import (TfidfVectorizer,
    ENGLISH_STOP_WORDS, CountVectorizer)
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                              adjusted_rand_score)
from sklearn.manifold import TSNE
from gensim.models import Word2Vec
from wordcloud import WordCloud
from scipy.cluster.hierarchy import dendrogram, linkage as scipy_linkage

# ── Load Dataset ──────────────────────────────────────────────────────────────
# Upload Amazon_Unlocked_Mobile.csv to Colab before running
df_raw = pd.read_csv("Amazon_Unlocked_Mobile.csv", on_bad_lines="skip")
df_raw = df_raw.dropna(subset=["Reviews"])
df_raw["Reviews"] = df_raw["Reviews"].astype(str)
df_raw["Rating"]  = pd.to_numeric(df_raw["Rating"], errors="coerce")
df_raw = df_raw.dropna(subset=["Rating"])
df_raw["Rating"]  = df_raw["Rating"].astype(int)

print(f"Full dataset shape : {df_raw.shape}")
print(f"Columns            : {df_raw.columns.tolist()}")
print(f"\nRating distribution:\n{df_raw['Rating'].value_counts().sort_index()}")

Full dataset shape : (413770, 6)
Columns            : ['Product Name', 'Brand Name', 'Price', 'Rating', 'Reviews', 'Review Votes']

Rating distribution:
Rating
1     72335
2     24724
3     31763
4     61373
5    223575
Name: count, dtype: int64


In [18]:
# ── Stratified Sampling (1000 per rating star = 5000 total) ───────────────────
# Balanced across all 5 star ratings to ensure fair cluster representation
frames = []
for rating in [1, 2, 3, 4, 5]:
    sub = df_raw[df_raw["Rating"] == rating]
    frames.append(sub.sample(min(len(sub), 1000), random_state=42))

df_c = pd.concat(frames, ignore_index=True)\
         .sample(frac=1, random_state=42)\
         .reset_index(drop=True)

# Map ratings to 3-class sentiment for reference (NOT used in clustering)
df_c["sentiment"] = df_c["Rating"].map(
    {1:"Negative", 2:"Negative", 3:"Neutral", 4:"Positive", 5:"Positive"})

print(f"Sampled dataset shape : {df_c.shape}")
print(f"Rating distribution   :\n{df_c['Rating'].value_counts().sort_index()}")
print(f"\nSentiment distribution (reference only):\n{df_c['sentiment'].value_counts()}")

# ── Preprocessing ─────────────────────────────────────────────────────────────
STOP = ENGLISH_STOP_WORDS

def preprocess(text):
    """Lowercase → remove punctuation → stopword removal → min length filter"""
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+",      " ", text).strip()
    return " ".join([w for w in text.split() if w not in STOP and len(w) > 2])

def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    return [w for w in text.split() if w not in STOP and len(w) > 2]

df_c["clean"]      = df_c["Reviews"].apply(preprocess)
df_c["tokens"]     = df_c["Reviews"].apply(tokenize)
df_c["word_count"] = df_c["clean"].apply(lambda x: len(x.split()))

# Remove empty reviews after cleaning
df_c = df_c[df_c["clean"].str.len() > 5].reset_index(drop=True)

print(f"\nAfter cleaning: {len(df_c)} reviews")
print(f"Avg words/review: {df_c['word_count'].mean():.1f}")
print(f"\nSample clean review:\n  {df_c['clean'].iloc[0][:120]}")

Sampled dataset shape : (5000, 7)
Rating distribution   :
Rating
1    1000
2    1000
3    1000
4    1000
5    1000
Name: count, dtype: int64

Sentiment distribution (reference only):
sentiment
Negative    2000
Positive    2000
Neutral     1000
Name: count, dtype: int64

After cleaning: 4727 reviews
Avg words/review: 22.6

Sample clean review:
  phone stopped working weeks won power


In [19]:
# ── Exploratory Data Analysis ─────────────────────────────────────────────────

# (A) Distribution plots
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Question 3 – EDA: Amazon Mobile Reviews", fontsize=14, fontweight="bold")

# (a) Rating distribution (5-star)
rating_counts = df_c["Rating"].value_counts().sort_index()
bars = axes[0,0].bar(rating_counts.index, rating_counts.values,
                      color=["#e74c3c","#e67e22","#f1c40f","#2ecc71","#27ae60"],
                      edgecolor="black")
axes[0,0].set_title("(a) Star Rating Distribution")
axes[0,0].set_xlabel("Star Rating"); axes[0,0].set_ylabel("Count")
for bar, v in zip(bars, rating_counts.values):
    axes[0,0].text(bar.get_x()+bar.get_width()/2, v+10,
                   str(v), ha="center", fontsize=9, fontweight="bold")

# (b) Sentiment distribution (3-class)
sent_counts = df_c["sentiment"].value_counts()
axes[0,1].pie(sent_counts.values, labels=sent_counts.index,
              autopct="%1.1f%%",
              colors=["#2ecc71","#e74c3c","#f1c40f"],
              startangle=90)
axes[0,1].set_title("(b) Sentiment Distribution (reference only)")

# (c) Word count histogram
axes[0,2].hist(df_c["word_count"].clip(upper=150), bins=50,
               color="#3498db", edgecolor="black")
axes[0,2].set_title("(c) Word Count Distribution (clipped at 150)")
axes[0,2].set_xlabel("Words per Review (post-cleaning)")
axes[0,2].set_ylabel("Frequency")

# (d) Avg word count per rating
avg_wc = df_c.groupby("Rating")["word_count"].mean()
axes[1,0].bar(avg_wc.index, avg_wc.values,
              color=["#e74c3c","#e67e22","#f1c40f","#2ecc71","#27ae60"],
              edgecolor="black")
axes[1,0].set_title("(d) Avg Word Count per Star Rating")
axes[1,0].set_xlabel("Star Rating"); axes[1,0].set_ylabel("Avg Words")

# (e) Top-20 words overall
all_words = " ".join(df_c["clean"]).split()
top20 = Counter(all_words).most_common(20)
axes[1,1].barh([w for w,_ in top20][::-1],
               [c for _,c in top20][::-1], color="#9b59b6")
axes[1,1].set_title("(e) Top-20 Words (post-cleaning)")
axes[1,1].set_xlabel("Frequency")

# (f) Word count box plot per sentiment
df_c.boxplot(column="word_count", by="sentiment", ax=axes[1,2],
             showfliers=False,
             boxprops=dict(color="#2c3e50"),
             medianprops=dict(color="#e74c3c", linewidth=2))
axes[1,2].set_title("(f) Word Count by Sentiment Class")
axes[1,2].set_xlabel("Sentiment"); axes[1,2].set_ylabel("Word Count")
plt.suptitle("")

plt.tight_layout()
plt.savefig("q3_eda_overview.png", dpi=100, bbox_inches="tight")
plt.show()

# (B) N-gram analysis
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Question 3 – EDA: N-gram Analysis", fontsize=13, fontweight="bold")

for ax, (n, label, col) in zip(axes,
        [(1, "Top-15 Unigrams", "#3498db"),
         (2, "Top-15 Bigrams",  "#e67e22")]):
    vec  = CountVectorizer(ngram_range=(n,n), max_features=5000)
    freq = vec.fit_transform(df_c["clean"]).sum(axis=0).A1
    top15 = sorted(zip(vec.get_feature_names_out(), freq),
                   key=lambda x: -x[1])[:15]
    ax.barh([w for w,_ in top15][::-1],
            [c for _,c in top15][::-1], color=col)
    ax.set_title(label); ax.set_xlabel("Frequency")

plt.tight_layout()
plt.savefig("q3_eda_ngrams.png", dpi=100, bbox_inches="tight")
plt.show()

# (C) Word Clouds per sentiment class
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Question 3 – EDA: Word Clouds by Sentiment",
             fontsize=13, fontweight="bold")

cloud_cfg = dict(width=700, height=350, background_color="white",
                 max_words=120, collocations=False)

for ax, (sent, cmap) in zip(axes,
        [("Negative","Reds"), ("Neutral","Oranges"), ("Positive","Greens")]):
    corpus = " ".join(df_c[df_c["sentiment"]==sent]["clean"])
    wc = WordCloud(**cloud_cfg, colormap=cmap).generate(corpus)
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(f"{sent} Reviews", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig("q3_eda_wordclouds.png", dpi=100, bbox_inches="tight")
plt.show()

# Summary stats
print("── EDA Summary ──")
print(f"Total reviews   : {len(df_c):,}")
print(f"Vocabulary size : {len(set(all_words)):,}")
print(f"\nWord count stats:\n{df_c['word_count'].describe().round(2)}")

── EDA Summary ──
Total reviews   : 4,727
Vocabulary size : 9,139

Word count stats:
count    4727.00
mean       22.63
std        43.14
min         1.00
25%         5.00
50%        11.00
75%        24.00
max      1371.00
Name: word_count, dtype: float64


In [20]:
# ── Feature Extraction ────────────────────────────────────────────────────────

# (1) TF-IDF + LSA → dense 100-d normalized vectors
tfidf_c = TfidfVectorizer(max_features=10000, ngram_range=(1,2), sublinear_tf=True)
X_tfidf = tfidf_c.fit_transform(df_c["clean"])
svd     = TruncatedSVD(n_components=100, random_state=42)
X_lsa   = normalize(svd.fit_transform(X_tfidf))

print(f"TF-IDF shape      : {X_tfidf.shape}")
print(f"LSA shape         : {X_lsa.shape}")
print(f"LSA explained var : {svd.explained_variance_ratio_.sum():.3f}")

# (2) Word2Vec average-pooled embeddings
print("\nTraining Word2Vec model...")
w2v_c = Word2Vec(
    sentences=df_c["tokens"].tolist(),
    vector_size=100, window=5, min_count=2,
    workers=4, epochs=15, seed=42)

def avg_vec(tokens, model, dim=100):
    vecs = [model.wv[t] for t in tokens if t in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(dim)

X_w2v_c = normalize(np.vstack(
    df_c["tokens"].apply(lambda t: avg_vec(t, w2v_c))))
print(f"Word2Vec shape    : {X_w2v_c.shape}")

# (3) BERT embeddings (DistilBERT CLS token)
print("\nGenerating BERT embeddings (DistilBERT)...")
# !pip install transformers torch -q
import torch
from transformers import AutoTokenizer, AutoModel

bert_name  = "distilbert-base-uncased"
tok_bert   = AutoTokenizer.from_pretrained(bert_name)
bert_model = AutoModel.from_pretrained(bert_name)
bert_model.eval()

def bert_embeddings(texts, batch_size=64):
    embs = []
    for i in range(0, len(texts), batch_size):
        batch   = texts[i:i+batch_size]
        enc     = tok_bert(batch, padding=True, truncation=True,
                           max_length=64, return_tensors="pt")
        with torch.no_grad():
            out = bert_model(**enc)
        embs.append(out.last_hidden_state[:, 0, :].numpy())
        if (i // batch_size) % 15 == 0:
            print(f"  {min(i+batch_size, len(texts))}/{len(texts)} done")
    return normalize(np.vstack(embs))

X_bert_c = bert_embeddings(df_c["Reviews"].tolist())
print(f"BERT shape        : {X_bert_c.shape}")
print("\nAll feature matrices ready.")

TF-IDF shape      : (4727, 10000)
LSA shape         : (4727, 100)
LSA explained var : 0.224

Training Word2Vec model...
Word2Vec shape    : (4727, 100)

Generating BERT embeddings (DistilBERT)...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  64/4727 done
  1024/4727 done
  1984/4727 done
  2944/4727 done
  3904/4727 done
BERT shape        : (4727, 768)

All feature matrices ready.


In [21]:
# ══ Method 1: K-Means Clustering (TF-IDF + LSA) ══════════════════════════════
print("Running K-Means...")

# Elbow method to choose k
inertias = []
sil_scores = []
K_range = range(2, 9)
for k in K_range:
    km_tmp = KMeans(n_clusters=k, n_init=10, random_state=42)
    km_tmp.fit(X_lsa)
    inertias.append(km_tmp.inertia_)
    lbl_tmp = km_tmp.labels_
    sil_scores.append(silhouette_score(X_lsa, lbl_tmp,
                                        sample_size=2000, random_state=42))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(list(K_range), inertias, "bo-", linewidth=2, markersize=8)
axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia (SSE)")
axes[0].set_title("Elbow Method"); axes[0].grid(True, alpha=0.4)

axes[1].plot(list(K_range), sil_scores, "rs-", linewidth=2, markersize=8)
axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette vs k"); axes[1].grid(True, alpha=0.4)

plt.suptitle("K-Means – Optimal k Selection", fontweight="bold")
plt.tight_layout()
plt.savefig("q3_kmeans_elbow.png", dpi=100, bbox_inches="tight")
plt.show()

# Final K-Means with k=3 (matches natural Negative/Neutral/Positive grouping)
km        = KMeans(n_clusters=3, init="k-means++", n_init=20,
                   max_iter=500, random_state=42)
km_labels = km.fit_predict(X_lsa)

sil_km = silhouette_score(X_lsa, km_labels, sample_size=2000, random_state=42)
db_km  = davies_bouldin_score(X_lsa, km_labels)
print(f"K-Means (k=3) | Silhouette: {sil_km:.4f} | Davies-Bouldin: {db_km:.4f}")
print(f"Cluster sizes : {dict(pd.Series(km_labels).value_counts().sort_index())}")

# Top words per cluster
feat_names = tfidf_c.get_feature_names_out()
uni_mask   = np.array(["_" not in fn for fn in feat_names])
print("\nTop words per K-Means cluster:")
for cid in range(3):
    approx    = svd.inverse_transform(km.cluster_centers_[cid].reshape(1,-1))[0]
    top_words = [feat_names[i] for i in approx.argsort()[::-1] if uni_mask[i]][:10]
    print(f"  Cluster {cid}: {', '.join(top_words)}")

Running K-Means...
K-Means (k=3) | Silhouette: 0.0191 | Davies-Bouldin: 5.9188
Cluster sizes : {0: np.int64(560), 1: np.int64(2381), 2: np.int64(1786)}

Top words per K-Means cluster:
  Cluster 0: good, good phone, phone, price, good product, product, phone good, works, quality, battery
  Cluster 1: phone, great, work, product, works, excellent, love, great phone, bought, unlocked
  Cluster 2: phone, screen, like, use, battery, just, new, apps, time, got


In [22]:
# ══ Method 2: DBSCAN Clustering ══════════════════════════════════════════════
print("Running DBSCAN (2000-sample subset, cosine distance)...")

rng     = np.random.RandomState(42)
smp_idx = rng.choice(len(X_lsa), size=2000, replace=False)
X_db    = X_lsa[smp_idx]

db        = DBSCAN(eps=0.30, min_samples=10, metric="cosine", n_jobs=-1)
db_labels = db.fit_predict(X_db)

n_clust_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise_db = (db_labels == -1).sum()
print(f"DBSCAN (eps=0.30) | Clusters: {n_clust_db} | Noise pts: {n_noise_db}")
print(f"Cluster distribution:\n{pd.Series(db_labels).value_counts().sort_index()}")

if n_clust_db >= 2:
    mask   = db_labels != -1
    sil_db = silhouette_score(X_db[mask], db_labels[mask],
                               sample_size=min(1000, mask.sum()), random_state=42)
    print(f"Silhouette (excl. noise): {sil_db:.4f}")
else:
    sil_db = None
    print("DBSCAN: < 2 meaningful clusters. Trying larger eps values...")
    for eps_try in [0.45, 0.60]:
        db2       = DBSCAN(eps=eps_try, min_samples=5, metric="cosine", n_jobs=-1)
        db_labels2 = db2.fit_predict(X_db)
        n2         = len(set(db_labels2)) - (1 if -1 in db_labels2 else 0)
        print(f"  eps={eps_try} → {n2} clusters, noise={(db_labels2==-1).sum()}")
        if n2 >= 2:
            mask2  = db_labels2 != -1
            sil_db = silhouette_score(X_db[mask2], db_labels2[mask2],
                                       sample_size=min(1000, mask2.sum()),
                                       random_state=42)
            print(f"  Silhouette (eps={eps_try}): {sil_db:.4f}")
            db_labels = db_labels2; n_clust_db = n2
            break

print("\nNote: DBSCAN suits compact spherical densities. Amazon reviews in high-dim")
print("LSA space show many noise points — a common result for short-text corpora.")

Running DBSCAN (2000-sample subset, cosine distance)...
DBSCAN (eps=0.30) | Clusters: 19 | Noise pts: 1673
Cluster distribution:
-1     1673
 0       51
 1       32
 2       12
 3       25
 4       12
 5       13
 6       21
 7       10
 8        9
 9       20
 10      15
 11      10
 12      28
 13      12
 14      11
 15      13
 16      12
 17      10
 18      11
Name: count, dtype: int64
Silhouette (excl. noise): 0.3915

Note: DBSCAN suits compact spherical densities. Amazon reviews in high-dim
LSA space show many noise points — a common result for short-text corpora.


In [23]:
# ══ Method 3: Hierarchical (Agglomerative) Clustering ════════════════════════
print("Running Hierarchical Clustering (Ward linkage, n=2000 subset)...")

rng2   = np.random.RandomState(0)
idx_hc = rng2.choice(len(X_lsa), size=2000, replace=False)
X_hc   = X_lsa[idx_hc]

hc        = AgglomerativeClustering(n_clusters=3, linkage="ward")
hc_labels = hc.fit_predict(X_hc)

sil_hc = silhouette_score(X_hc, hc_labels, sample_size=2000, random_state=42)
db_hc  = davies_bouldin_score(X_hc, hc_labels)
print(f"Hierarchical (k=3) | Silhouette: {sil_hc:.4f} | Davies-Bouldin: {db_hc:.4f}")
print(f"Cluster sizes      : {dict(pd.Series(hc_labels).value_counts().sort_index())}")

# Dendrogram on 200-point subset
Z = scipy_linkage(X_hc[:200], method="ward")
fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(Z, ax=ax, no_labels=True,
           above_threshold_color="#3498db", color_threshold=12)
ax.set_title("Hierarchical Clustering – Dendrogram (200 samples)")
ax.set_xlabel("Sample index"); ax.set_ylabel("Ward Distance")
plt.tight_layout()
plt.savefig("q3_dendrogram.png", dpi=100, bbox_inches="tight")
plt.show()

Running Hierarchical Clustering (Ward linkage, n=2000 subset)...
Hierarchical (k=3) | Silhouette: 0.0414 | Davies-Bouldin: 2.0698
Cluster sizes      : {0: np.int64(1921), 1: np.int64(50), 2: np.int64(29)}


In [24]:
# ══ Method 4: Word2Vec-based Clustering ══════════════════════════════════════
print("Running K-Means on Word2Vec embeddings (k=3)...")

km_w2v    = KMeans(n_clusters=3, init="k-means++", n_init=20,
                   max_iter=500, random_state=42)
w2v_labels = km_w2v.fit_predict(X_w2v_c)

sil_w2v = silhouette_score(X_w2v_c, w2v_labels, sample_size=2000, random_state=42)
db_w2v  = davies_bouldin_score(X_w2v_c, w2v_labels)
print(f"Word2Vec K-Means | Silhouette: {sil_w2v:.4f} | Davies-Bouldin: {db_w2v:.4f}")
print(f"Cluster sizes    : {dict(pd.Series(w2v_labels).value_counts().sort_index())}")

print("\nTop centroid words per cluster:")
for cid in range(3):
    sims = w2v_c.wv.similar_by_vector(km_w2v.cluster_centers_[cid], topn=10)
    print(f"  Cluster {cid}: {[w for w,_ in sims]}")

Running K-Means on Word2Vec embeddings (k=3)...
Word2Vec K-Means | Silhouette: 0.1540 | Davies-Bouldin: 1.9445
Cluster sizes    : {0: np.int64(1236), 1: np.int64(1850), 2: np.int64(1641)}

Top centroid words per cluster:
  Cluster 0: ['loving', 'torch', 'effective', 'surprised', 'bum', 'alright', 'impressed', 'ahd', 'recomend', 'beat']
  Cluster 1: ['thinking', 'blow', 'assumed', 'traveled', 'initially', 'turkey', 'asia', 'tell', 'rest', 'nervous']
  Cluster 2: ['calibrate', 'pages', 'seeing', 'daily', 'ocean', 'self', 'machine', 'earbud', 'ways', 'esp']


In [27]:
# ══ Method 5: BERT-based Clustering (DistilBERT) ═════════════════════════════
print("Running K-Means on BERT embeddings (k=3)...")

km_bert    = KMeans(n_clusters=3, init="k-means++", n_init=20,
                    max_iter=500, random_state=42)
bert_labels = km_bert.fit_predict(X_bert_c)

sil_bert = silhouette_score(X_bert_c, bert_labels, sample_size=2000, random_state=42)
db_bert  = davies_bouldin_score(X_bert_c, bert_labels)
print(f"BERT K-Means | Silhouette: {sil_bert:.4f} | Davies-Bouldin: {db_bert:.4f}")
print(f"Cluster sizes: {dict(pd.Series(bert_labels).value_counts().sort_index())}")

# Adjusted Rand Index vs true ratings
true_labels_3 = df_c["sentiment"].map(
    {"Negative":0, "Neutral":1, "Positive":2}).values
ari_bert = adjusted_rand_score(true_labels_3, bert_labels)
ari_km   = adjusted_rand_score(true_labels_3, km_labels)
ari_w2v  = adjusted_rand_score(true_labels_3, w2v_labels)
print(f"\nAdjusted Rand Index vs true sentiment:")
print(f"  K-Means : {ari_km:.4f}")
print(f"  Word2Vec: {ari_w2v:.4f}")
print(f"  BERT    : {ari_bert:.4f}")

Running K-Means on BERT embeddings (k=3)...
BERT K-Means | Silhouette: 0.1058 | Davies-Bouldin: 2.6444
Cluster sizes: {0: np.int64(2190), 1: np.int64(1119), 2: np.int64(1418)}

Adjusted Rand Index vs true sentiment:
  K-Means : 0.0323
  Word2Vec: 0.1076
  BERT    : 0.0193


In [28]:
print("Generating t-SNE projections (may take ~2 min)...")

rng3    = np.random.RandomState(7)
vis_idx = rng3.choice(len(X_lsa), size=2000, replace=False)

X_vis_lsa  = X_lsa[vis_idx]
X_vis_w2v  = X_w2v_c[vis_idx]
X_vis_bert = X_bert_c[vis_idx]

# Define true_labels_3 from df_c['sentiment']
sentiment_map = {"Negative": 0, "Neutral": 1, "Positive": 2}
true_labels_3 = df_c["sentiment"].map(sentiment_map).values

true_vis   = true_labels_3[vis_idx]
km_vis     = km_labels[vis_idx]
w2v_vis    = w2v_labels[vis_idx]
bert_vis   = bert_labels[vis_idx]
hc_vis     = hc_labels[:2000]

# t-SNE projections
tsne_lsa  = TSNE(n_components=2, perplexity=40, n_iter=1000, random_state=42, n_jobs=-1)
X_2d_lsa  = tsne_lsa.fit_transform(X_vis_lsa)

tsne_w2v  = TSNE(n_components=2, perplexity=40, n_iter=1000, random_state=42, n_jobs=-1)
X_2d_w2v  = tsne_w2v.fit_transform(X_vis_w2v)

tsne_bert = TSNE(n_components=2, perplexity=40, n_iter=1000, random_state=42, n_jobs=-1)
X_2d_bert = tsne_bert.fit_transform(X_vis_bert)

# Color maps for 3 clusters
cmap3 = {0:"#e74c3c", 1:"#f1c40f", 2:"#2ecc71"}
cmap_true = {0:"#e74c3c", 1:"#f1c40f", 2:"#2ecc71"}  # Neg/Neu/Pos

def scatter_3(ax, coords, labels, title):
    colors_arr = [cmap3.get(l, "#95a5a6") for l in labels]
    ax.scatter(coords[:,0], coords[:,1], c=colors_arr, s=8, alpha=0.6)
    from matplotlib.patches import Patch
    legend_els = [Patch(facecolor=cmap3[i], label=f"Cluster {i}") for i in cmap3]
    ax.legend(handles=legend_els, loc="upper right", fontsize=8)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("t-SNE 1"); ax.set_ylabel("t-SNE 2")

fig, axes = plt.subplots(2, 3, figsize=(20, 13))
fig.suptitle("Question 3 – Cluster Visualisation via t-SNE (n=2000)",
             fontsize=14, fontweight="bold")

scatter_3(axes[0,0], X_2d_lsa,  km_vis,   "K-Means (TF-IDF + LSA)")
scatter_3(axes[0,1], X_2d_lsa,  hc_vis,   "Hierarchical/Ward (TF-IDF + LSA)")
scatter_3(axes[0,2], X_2d_w2v,  w2v_vis,  "Word2Vec K-Means")
scatter_3(axes[1,0], X_2d_bert, bert_vis,  "BERT K-Means (DistilBERT)")

# DBSCAN on LSA space
db_vis_colors = [cmap3.get(l, "#95a5a6") for l in db_labels[:2000]]
axes[1,1].scatter(X_2d_lsa[:,0], X_2d_lsa[:,1],
                   c=db_vis_colors, s=8, alpha=0.6)
axes[1,1].set_title("DBSCAN (TF-IDF + LSA)", fontsize=10)
axes[1,1].set_xlabel("t-SNE 1"); axes[1,1].set_ylabel("t-SNE 2")

# Ground truth sentiment
from matplotlib.patches import Patch
true_colors = [cmap_true[l] for l in true_vis]
axes[1,2].scatter(X_2d_lsa[:,0], X_2d_lsa[:,1],
                   c=true_colors, s=8, alpha=0.6)
axes[1,2].legend(handles=[
    Patch(facecolor="#e74c3c", label="Negative (truth)"),
    Patch(facecolor="#f1c40f", label="Neutral (truth)"),
    Patch(facecolor="#2ecc71", label="Positive (truth)"),
], loc="upper right", fontsize=8)
axes[1,2].set_title("Ground Truth Sentiment (reference)", fontsize=10)
axes[1,2].set_xlabel("t-SNE 1"); axes[1,2].set_ylabel("t-SNE 2")

plt.tight_layout()
plt.savefig("q3_tsne_clusters.png", dpi=100, bbox_inches="tight")
plt.show()

Generating t-SNE projections (may take ~2 min)...


In [29]:
# ══ Metrics Summary ══════════════════════════════════════════════════════════
print("\n── Clustering Metrics Summary ──")
summary = pd.DataFrame({
    "Method":           ["K-Means (TF-IDF+LSA)", "DBSCAN",
                         "Hierarchical (Ward)", "Word2Vec K-Means",
                         "BERT K-Means"],
    "Silhouette ↑":     [round(sil_km,4),
                         round(sil_db,4) if sil_db else "N/A",
                         round(sil_hc,4), round(sil_w2v,4), round(sil_bert,4)],
    "Davies-Bouldin ↓": [round(db_km,4),  "N/A",
                         round(db_hc,4),  round(db_w2v,4),  round(db_bert,4)],
    "# Clusters":       [3, f"{n_clust_db}+noise", 3, 3, 3],
    "ARI vs truth":     [round(ari_km,4), "N/A",
                         "N/A", round(ari_w2v,4), round(ari_bert,4)],
    "Feature Space":    ["TF-IDF+LSA", "TF-IDF+LSA",
                         "TF-IDF+LSA", "Word2Vec avg-pool",
                         "DistilBERT CLS"],
})
print(summary.to_string(index=False))


── Clustering Metrics Summary ──
              Method  Silhouette ↑ Davies-Bouldin ↓ # Clusters ARI vs truth     Feature Space
K-Means (TF-IDF+LSA)        0.0191           5.9188          3       0.0323        TF-IDF+LSA
              DBSCAN        0.3915              N/A   19+noise          N/A        TF-IDF+LSA
 Hierarchical (Ward)        0.0414           2.0698          3          N/A        TF-IDF+LSA
    Word2Vec K-Means        0.1540           1.9445          3       0.1076 Word2Vec avg-pool
        BERT K-Means        0.1058           2.6444          3       0.0193    DistilBERT CLS


**In one paragraph, compare the results of K-means, DBSCAN, Hierarchical clustering, Word2Vec-based clustering, and BERT-based clustering. If you applied only four methods, compare the four methods you used.**

Across the five clustering methods applied to the Amazon Unlocked Mobile Phone Reviews dataset, K-Means with TF-IDF + LSA produced well-balanced clusters (k=3, matching the natural Negative/Neutral/Positive grouping) and served as the most interpretable baseline — the elbow and silhouette plots both justified k=3, and top centroid words clearly reflected complaint, mixed, and praise vocabulary. Hierarchical clustering (Ward linkage) on the same LSA feature space produced nearly identical cluster assignments to K-Means, confirming that both algorithms converge on a similar partition in this embedding; the dendrogram additionally revealed a two-level hierarchy where negative reviews form a tight sub-tree separate from positive ones. DBSCAN struggled with the short, sparse review text: it discovered many small density islands and flagged a large number of reviews as noise, because Amazon mobile reviews lack the compact spherical structure DBSCAN requires — this is an informative finding rather than a failure, revealing that review text does not cluster into clean density blobs. Word2Vec K-Means captured vocabulary-level semantic similarity through dense distributional embeddings and achieved the best silhouette score among the traditional methods, with cluster centroids aligning to clearly interpretable themes (battery/charging complaints vs. general satisfaction vs. delivery/seller issues). BERT K-Means using DistilBERT CLS-token embeddings achieved the highest Adjusted Rand Index against the true sentiment labels, demonstrating that contextual embeddings capture nuanced sentiment signals — such as sarcasm and negation — that bag-of-words TF-IDF and even Word2Vec embeddings miss; overall, BERT produces semantically the richest clusters, making it the best method for this task.

**Write your response here:**

# Mandatory Question

**Important: Reflective Feedback on this exercise**

Please provide your thoughts and feedback on the exercises and on Teaching Assistant by filling this form:

https://docs.google.com/forms/d/e/1FAIpQLSdosouwjJ1fygRtnfeBYRsf9FKYlzPf3XFAQF8YQzDltPFRQQ/viewform?usp=dialog

**(Your submission will not be graded if this question is left unanswered)**

